# Notebook 12 — Human Genetics and Disease

**Module:** 05 — Biology Fundamentals  
**Notebook:** 12 of 18  
**Prerequisites:** NB05, NB06  
**Time estimate:** 75 minutes

> **Track A critical.** Pedigree analysis, carrier frequency calculation,
> and OMIM-level disease genetics appear in bioinformatics interviews.

---
## Step 1 — Motivation

Clinical bioinformatics — variant interpretation, pharmacogenomics, cancer genomics —
requires understanding how genetic variants cause disease. When GATK calls a variant,
understanding whether it's pathogenic requires knowing the gene's inheritance mode,
the population frequency, and the biological consequence.

---
## Step 3 — Biological Background

**Monogenic (single-gene) diseases:**

| Disease | Gene | Inheritance | Prevalence |
|---------|------|-------------|------------|
| Cystic fibrosis | CFTR | Autosomal recessive | 1/2,500 (European) |
| Huntington's disease | HTT | Autosomal dominant | 1/10,000 |
| Haemophilia A | F8 | X-linked recessive | 1/5,000 males |
| BRCA1/2 | BRCA1/2 | Autosomal dominant (incomplete penetrance) | 1/300–500 |
| Sickle-cell anemia | HBB | Autosomal recessive | 1/500 (African ancestry) |

**Polygenic diseases (GWAS targets):**
Most common diseases (T2D, schizophrenia, CAD) are influenced by thousands of variants,
each with a small effect. GWAS finds these variants. Polygenic risk scores (PRS)
sum their effects to predict individual risk.

**Variant classification (ClinVar/ACMG):**
- **Pathogenic:** causes disease
- **Likely pathogenic:** strong evidence of disease cause
- **Uncertain significance (VUS):** unclear
- **Likely benign / Benign:** does not cause disease

**Loss of function vs. gain of function:**
- LoF mutations disable the protein (frameshift, nonsense, splice site)
- GoF mutations create a new harmful function (e.g., oncogenic KRAS G12D)

**Penetrance and expressivity:**
- **Penetrance:** fraction of people with the genotype who show the phenotype
  (BRCA1 has ~70% lifetime penetrance for breast cancer)
- **Expressivity:** how severely the phenotype is expressed in those who do show it

---
## Step 4 — Mathematical Explanation

**Carrier frequency from disease prevalence (via HWE):**
For an autosomal recessive disease:
- Disease frequency = q²
- Allele frequency q = √(disease frequency)
- Carrier frequency = 2pq ≈ 2q (when q << 1)

**Allele count in gnomAD / population genetics:**
- Allele count (AC): number of copies of the allele in the cohort
- Allele number (AN): total number of alleles observed (2 × number of individuals)
- Allele frequency (AF) = AC / AN
- Filter: pathogenic variants for rare diseases usually have AF < 1%

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Carrier frequency calculator
def carrier_frequency_from_prevalence(disease_prevalence: float) -> dict:
    """
    Given autosomal recessive disease prevalence, compute allele and carrier frequencies.
    Assumes Hardy-Weinberg equilibrium.
    """
    q = np.sqrt(disease_prevalence)    # disease freq = q²
    p = 1.0 - q
    carrier_freq = 2 * p * q
    return {
        'disease_prevalence': disease_prevalence,
        'allele_freq_q': q,
        'carrier_freq': carrier_freq,
        'fraction_carriers_vs_affected': carrier_freq / disease_prevalence,
    }

diseases = [
    ('Cystic fibrosis (European)', 1/2500),
    ('PKU', 1/10000),
    ('Sickle-cell anemia', 1/400),
    ('Galactosemia', 1/60000),
]

print(f"{'Disease':<35} {'Prevalence':>12} {'q (AF)':>10} {'Carrier freq':>14} {'Carriers/Affected':>18}")
print("-" * 95)
for name, prev in diseases:
    r = carrier_frequency_from_prevalence(prev)
    print(f"  {name:<33} {prev:>12.1e} {r['allele_freq_q']:>10.4f} "
          f"{r['carrier_freq']:>14.4f} {r['fraction_carriers_vs_affected']:>18.1f}×")

In [ ]:
# Cell 6.2 — Risk of affected child given parental carrier status
def affected_child_risk(
    parent1_carrier: bool,
    parent2_carrier: bool,
    inheritance: str = 'autosomal_recessive'
) -> float:
    """
    Risk of an affected child given parental carrier status.
    """
    if inheritance == 'autosomal_recessive':
        if parent1_carrier and parent2_carrier:
            return 0.25   # Aa × Aa → 1/4 aa
        elif parent1_carrier or parent2_carrier:
            return 0.0    # Aa × AA → no affected children
        else:
            return 0.0    # AA × AA → no affected children
    elif inheritance == 'autosomal_dominant':
        if parent1_carrier or parent2_carrier:
            return 0.50   # Aa × aa → 50% children affected
        return 0.0
    else:
        raise ValueError(f"Unknown inheritance: {inheritance}")

print("Carrier risk scenarios for autosomal recessive disease:")
scenarios = [
    ('Both parents carriers',       True, True),
    ('One carrier, one non-carrier', True, False),
    ('Neither carrier',             False, False),
]
for desc, c1, c2 in scenarios:
    risk = affected_child_risk(c1, c2)
    print(f"  {desc:<35}: {risk:.0%} per child")

In [ ]:
# Cell 6.3 — Polygenic risk score simulation
rng = np.random.default_rng(42)

n_snps = 100       # number of GWAS variants
n_individuals = 5000

# Effect sizes (log OR) drawn from normal distribution (most are tiny)
effect_sizes = rng.normal(0, 0.05, size=n_snps)
# Allele frequencies uniform between 0.05 and 0.95
allele_freqs = rng.uniform(0.05, 0.95, size=n_snps)

# Genotypes: each individual has 0/1/2 copies of each risk allele
# (binomial with n=2, p=af for each SNP)
genotypes = rng.binomial(2, allele_freqs, size=(n_individuals, n_snps))

# PRS: sum of (effect_size × genotype) across SNPs
prs = genotypes @ effect_sizes

# Assign 'disease': individuals in top 10% PRS have 3× higher disease rate
prs_threshold = np.percentile(prs, 90)
high_risk = prs >= prs_threshold
disease_base_rate = 0.05
disease = rng.binomial(1, np.where(high_risk, disease_base_rate*3, disease_base_rate))

print(f"Polygenic risk score simulation: {n_snps} variants, {n_individuals} individuals")
print(f"  Mean PRS: {prs.mean():.3f}  SD: {prs.std():.3f}")
print(f"  Disease rate in bottom 90%: {disease[~high_risk].mean():.1%}")
print(f"  Disease rate in top 10%:    {disease[high_risk].mean():.1%}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Carrier frequency vs. prevalence + PRS distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Carrier frequency vs. disease prevalence
ax = axes[0]
prev_range = np.logspace(-5, -1, 200)
cf = [2 * (1-np.sqrt(p)) * np.sqrt(p) for p in prev_range]
ax.semilogx(prev_range, cf, color='steelblue', lw=2)

for name, prev in diseases:
    r = carrier_frequency_from_prevalence(prev)
    ax.plot(prev, r['carrier_freq'], 'o', ms=8, label=f"{name.split('(')[0].strip()}",
            zorder=5)

ax.set_xlabel('Disease prevalence (log scale)')
ax.set_ylabel('Carrier frequency 2pq')
ax.set_title('Carrier frequency vs. disease prevalence\n(autosomal recessive, HWE)')
ax.legend(fontsize=7)

# Panel 2: PRS distribution
ax = axes[1]
ax.hist(prs[~high_risk], bins=50, color='steelblue', alpha=0.6, label='Low risk (bottom 90%)')
ax.hist(prs[high_risk],  bins=20, color='tomato',    alpha=0.8, label='High risk (top 10%)')
ax.axvline(prs_threshold, color='gray', ls='--', lw=1.5, label=f'90th percentile')
ax.set_xlabel('Polygenic Risk Score')
ax.set_ylabel('Count')
ax.set_title(f'PRS distribution: {n_snps} variants\nTop 10% has {disease[high_risk].mean():.1%} disease rate')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `carrier_frequency_from_prevalence(disease_prevalence)` from scratch.
   Calculate the carrier frequency for spinal muscular atrophy (SMA), which has
   a prevalence of approximately 1/6,000.
2. A family has three children with an autosomal recessive disease, and both parents
   are unaffected. Is this consistent with Mendelian ratios? Calculate the probability
   of all three children being affected given that both parents are carriers.
3. What is the difference between penetrance and expressivity? Give one example of a
   disease where penetrance is incomplete.
4. A variant has AF = 0.3% in gnomAD (general population) and AF = 5% in a disease
   cohort. Calculate the odds ratio. Does this suggest a pathogenic variant?

---
## Quiz — Active Recall

1. What is the carrier frequency for cystic fibrosis (prevalence 1/2500) under HWE?
2. What is the difference between a monogenic and polygenic disease? Give one example
   of each.
3. What does 'loss of function' mean for a variant? How does it differ from 'gain of function'?
4. What is penetrance? Give a real example of incomplete penetrance.
5. A variant is classified as VUS (variant of uncertain significance). What does this mean?
   What evidence would you look for to reclassify it?

---
## Reflection

**Date completed:** ____________________

> *[A patient is found to carry a BRCA1 variant with ~70% penetrance for breast cancer.
> How would you explain 'penetrance' to the patient in plain language? What does
> 70% actually mean for their specific risk?]*

---
*Next: `13_reading_biology_figures.ipynb`*